# CIFAR-100 with `Dataset_vvtk` + `torch.utils.data.DataLoader`

This notebook uses the legacy `Dataset_vvtk` class from `old/dataset_vvtk.py`.

Workflow:
1. Download the CIFAR-100 training split with `torchvision`
2. Store **only the images** in a **single** `.vvtk` file
3. Save labels separately in a `.npy` file
4. Wrap both with a custom PyTorch `Dataset`
5. Train a small CNN for a few epochs with `torch.utils.data.DataLoader`

In [1]:
from pathlib import Path
import os
import sys

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'old' / 'dataset_vvtk.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root containing old/dataset_vvtk.py')


REPO_ROOT = find_repo_root(Path.cwd())
OLD_DIR = REPO_ROOT / 'old'
if str(OLD_DIR) not in sys.path:
    sys.path.insert(0, str(OLD_DIR))

from dataset_vvtk import Dataset_vvtk


DATA_DIR = OLD_DIR / 'data'
RAW_DIR = DATA_DIR / 'data_raw'
IMAGE_FILE = DATA_DIR / 'cifar100_train_images.vvtk'
LABEL_FILE = DATA_DIR / 'cifar100_train_labels.npy'
DATA_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(0)
np.random.seed(0)

BATCH_SIZE = 256
NUM_EPOCHS = 3
VAL_FRACTION = 0.1
MAX_SAMPLES = None  # set to an integer like 10000 for a faster demo
FORCE_REWRITE = False
NUM_WORKERS = min(4, os.cpu_count() or 1)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Repo root:   {REPO_ROOT}')
print(f'Data dir:    {DATA_DIR}')
print(f'Device:      {DEVICE}')
print(f'Num workers: {NUM_WORKERS}')

Repo root:   /extra3/cadrete/work/src/vvtk_dataset
Data dir:    /extra3/cadrete/work/src/vvtk_dataset/old/data
Device:      cuda
Num workers: 4


In [2]:
cifar100_train = datasets.CIFAR100(root=str(RAW_DIR), train=True, download=True)
num_total = len(cifar100_train) if MAX_SAMPLES is None else min(MAX_SAMPLES, len(cifar100_train))
sample_img, sample_label = cifar100_train[0]

print(f'Train samples available: {len(cifar100_train)}')
print(f'Samples used in this notebook: {num_total}')
print(f'Example image shape (HWC): {np.asarray(sample_img).shape}')
print(f'Example label: {sample_label}')

Train samples available: 50000
Samples used in this notebook: 50000
Example image shape (HWC): (32, 32, 3)
Example label: 19


In [3]:
def write_images_only_vvtk(torchvision_dataset, image_file: Path, label_file: Path, limit=None, force=False):
    total = len(torchvision_dataset) if limit is None else min(limit, len(torchvision_dataset))

    if not force and image_file.exists() and label_file.exists():
        labels = np.load(label_file)
        if len(labels) == total:
            print(f'Using existing files: {image_file.name} and {label_file.name}')
            return labels

    if image_file.exists():
        image_file.unlink()
    if label_file.exists():
        label_file.unlink()

    writer = Dataset_vvtk(str(image_file), mode='wb', debug=False)
    labels = np.empty(total, dtype=np.int64)

    for idx in range(total):
        img_pil, label = torchvision_dataset[idx]
        img = np.asarray(img_pil, dtype=np.uint8).transpose(2, 0, 1)
        img = np.ascontiguousarray(img)
        writer.add(img, f'img_{idx:06d}')
        labels[idx] = label

        if (idx + 1) % 5000 == 0 or idx + 1 == total:
            print(f'Wrote {idx + 1}/{total} images')

    writer.close()
    np.save(label_file, labels)
    print(f'Saved image file: {image_file}')
    print(f'Saved labels file: {label_file}')
    return labels


labels = write_images_only_vvtk(
    cifar100_train,
    IMAGE_FILE,
    LABEL_FILE,
    limit=MAX_SAMPLES,
    force=FORCE_REWRITE,
)

print(f'Stored {len(labels)} labels separately from the image data file.')

Using existing files: cifar100_train_images.vvtk and cifar100_train_labels.npy
Stored 50000 labels separately from the image data file.


## Custom PyTorch dataset

The dataset lazily opens `Dataset_vvtk` per worker process. Images come from the `.vvtk` file, while labels are loaded from the `.npy` file.

In [4]:
class CIFAR100ImagesFromVVTK(Dataset):
    def __init__(self, image_file, label_file):
        self.image_file = Path(image_file)
        self.labels = np.load(label_file).astype(np.int64)
        self._storage = Dataset_vvtk(str(self.image_file), mode='rb', debug=False)
       

    def __len__(self):
        return len(self.labels)


    def __getitem__(self, idx):
        img = self._storage.get(f'img_{idx:06d}')
        img = torch.from_numpy(img.astype(np.float32)).div(255.0)
        label = int(self.labels[idx])
        return img, label


full_dataset = CIFAR100ImagesFromVVTK(IMAGE_FILE, LABEL_FILE)
val_size = max(1, int(len(full_dataset) * VAL_FRACTION))
train_size = len(full_dataset) - val_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(0),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda'),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda'),
)

images, labels_batch = next(iter(train_loader))
print(f'Train subset size: {len(train_dataset)}')
print(f'Val subset size:   {len(val_dataset)}')
print(f'Batch image shape:  {tuple(images.shape)}')
print(f'Batch label shape:  {tuple(labels_batch.shape)}')
print(f'Image dtype:        {images.dtype}')

  file_data: /extra3/cadrete/work/src/vvtk_dataset/old/data/cifar100_train_images.vvtk



Train subset size: 45000
Val subset size:   5000
Batch image shape:  (256, 3, 32, 32)
Batch label shape:  (256,)
Image dtype:        torch.float32


In [5]:
class SmallCIFAR100CNN(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = SmallCIFAR100CNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(model)
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

SmallCIFAR100CNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2048, out_features=256, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.

In [6]:
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for imgs, labels_batch in train_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels_batch = labels_batch.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)
        train_correct += (logits.argmax(dim=1) == labels_batch).sum().item()
        train_total += imgs.size(0)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for imgs, labels_batch in val_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels_batch = labels_batch.to(DEVICE, non_blocking=True)

            logits = model(imgs)
            loss = criterion(logits, labels_batch)

            val_loss += loss.item() * imgs.size(0)
            val_correct += (logits.argmax(dim=1) == labels_batch).sum().item()
            val_total += imgs.size(0)

    print(
        f'Epoch {epoch:02d}/{NUM_EPOCHS} | ' 
        f'train loss: {train_loss / train_total:.4f} | ' 
        f'train acc: {100.0 * train_correct / train_total:.2f}% | ' 
        f'val loss: {val_loss / val_total:.4f} | ' 
        f'val acc: {100.0 * val_correct / val_total:.2f}%'
    )

Epoch 01/3 | train loss: 4.2710 | train acc: 4.67% | val loss: 3.9029 | val acc: 10.12%
Epoch 02/3 | train loss: 3.7095 | train acc: 12.81% | val loss: 3.3602 | val acc: 19.04%
Epoch 03/3 | train loss: 3.3775 | train acc: 18.48% | val loss: 3.1091 | val acc: 23.94%
